# **Weight Calculation Pipeline**
A semantic weighting system that transforms lemmatised tokens into weighted category vectors for NLP analysis.

## Overview
Second stage of a two-part preprocessing pipeline. Takes lemmatised text and assigns semantic weights to each token using a dictionary-based approach, creating dense vector representations for downstream machine learning tasks.

## Key Features


*   **Efficient processing**: NumPy-based vectorisation for high-performance weight calculation
*   **Dictionary mapping**: converts tokens to (category, weight) pairs
*   **Memory-optimised**: float32 precision and pre-allocated arrays for large datasets
*   **Flexible output**: optional filtering of zero-weight comments

## Core Components

*   load_dictionary(): Loads word-category-weight mappings from CSV
*   process_comments(): Converts token lists to weight matrices with progress tracking
*   Main pipeline: End-to-end processing with configurable arguments


## Input Requirements
*   **Dictionary CSV**: columns word, category, weight
*   **Input CSV**: must contain lemmatisation_with_tokens column (output from Text Lemmatisation Pipeline)

## Output
Generates enriched DataFrame with:
*   weight_{category} columns for each semantic category
*   total_weight column (sum of all weights per comment)
*   Optional filtering to remove zero-weight comments

## Technical Details
Built with pandas, NumPy, and tqdm. Optimised for processing comments through local variable caching and efficient dictionary lookups.

## Dependencies
pandas, numpy, tqdm, logging, argparse

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!wget https://raw.githubusercontent.com/ValeriaMagnatka/surprise_dataset/main/categories.csv

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import logging
import argparse
import sys

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def load_dictionary(file_path):
    """Load dictionary: word -> (category, weight)"""
    logger.info("Loading dictionary...")
    df = pd.read_csv(file_path)
    df = df.drop_duplicates('word', keep='first')
    word_info = dict(zip(df['word'], zip(df['category'], df['weight'])))
    categories = sorted(df['category'].unique())
    logger.info(f"Loaded {len(word_info)} words, {len(categories)} categories")
    return word_info, categories


def process_comments(df, word_info, categories):
    """Processing for comments"""
    logger.info("Processing comments...")

    # Split tokens
    tokens_list = df['lemmatisation_with_tokens'].str.split().tolist()

    n_comments = len(tokens_list)
    n_categories = len(categories)
    weights_matrix = np.zeros((n_comments, n_categories), dtype=np.float32)
    total_weights = np.zeros(n_comments, dtype=np.float32)

    cat_to_idx = {cat: i for i, cat in enumerate(categories)}

    # Optimisation: local bindings for speed
    word_info_get = word_info.get
    cat_to_idx_get = cat_to_idx.get

    for i, tokens in enumerate(tqdm(tokens_list, desc="Processing")):
        if not tokens:
            continue

        temp_weights = {}
        total = 0.0

        for token in tokens:
            info = word_info_get(token)  # single dictionary lookup
            if info:
                cat, weight = info
                temp_weights[cat] = temp_weights.get(cat, 0.0) + weight
                total += weight

        if total > 0:
            total_weights[i] = total
            for cat, w in temp_weights.items():
                weights_matrix[i, cat_to_idx_get(cat)] = w

    weights_df = pd.DataFrame(
        weights_matrix,
        columns=[f'weight_{cat}' for cat in categories]
    )
    weights_df['total_weight'] = total_weights

    # Combine original data with weights
    result = pd.concat([df, weights_df], axis=1)

    return result


def main():
    # Remove -f argument that Colab automatically adds
    if '-f' in sys.argv:
        idx = sys.argv.index('-f')
        sys.argv.pop(idx)
        if idx < len(sys.argv):
            sys.argv.pop(idx)  # Remove the path after -f

    if len(sys.argv) == 1:
        sys.argv.extend([
            '--input', 'geo-reviews-dataset-2023_11-30_lemmatised.csv'
        ])

    parser = argparse.ArgumentParser(description='Add weights to comments based on dictionary')
    parser.add_argument('--input', help='Input CSV file path')
    parser.add_argument('--dict', default='categories.csv', help='Dictionary CSV file path')
    parser.add_argument('--output', default='weighted_comments.csv', help='Output CSV file path')
    parser.add_argument('--save-all', action='store_true', help='Save all comments (including zero weights)')

    args = parser.parse_args()

    # Load dictionary
    word_info, categories = load_dictionary(args.dict)

    # Load comments
    logger.info(f"Loading comments from {args.input}...")
    df = pd.read_csv(args.input)
    logger.info(f"Loaded {len(df)} comments")

    # Check required column
    if 'lemmatisation_with_tokens' not in df.columns:
        raise ValueError("Column 'lemmatisation_with_tokens' not found in input file")

    # Process comments
    result_df = process_comments(df, word_info, categories)

    # Filter
    if not args.save_all:
        initial_len = len(result_df)
        result_df = result_df[result_df['total_weight'] > 0]
        logger.info(f"Kept {len(result_df)} comments with non-zero weights (filtered out {initial_len - len(result_df)})")
    else:
        logger.info(f"Keeping all {len(result_df)} comments (including zero weights)")
    # Save results

    result_df.to_csv(args.output, index=False)
    logger.info(f"Saved to {args.output}")



if __name__ == "__main__":
    main()